# PyTorch Reference for Week 2: CNNs & Data Loading

**Quick reference guide for PyTorch concepts used in Week 2**

This notebook covers PyTorch components that appear in Week 2's CNN labs but weren't covered in Week 1. Use this as a quick lookup reference while working through the main tutorial.

**Not sure about something?** Jump to the relevant section below.

**Want hands-on practice?** See the main tutorial: [week_02_cnns_rnns.ipynb](week_02_cnns_rnns.ipynb)

## Table of Contents

1. [Environment Setup](#environment-setup)
2. [Section 1: Datasets & DataLoaders](#section-1-datasets-dataloaders)
3. [Section 2: Image Transforms](#section-2-image-transforms)
4. [Section 3: CNN Layers](#section-3-cnn-layers)
5. [Section 4: Pretrained Models & Transfer Learning](#section-4-pretrained-models)
6. [Additional Resources](#additional-resources)

## Environment Setup

This notebook requires PyTorch and torchvision. Run the following cell to install dependencies (if using Google Colab).

In [ ]:
# Install required packages (Google Colab)
# Skip this cell if running locally with packages already installed

!pip install torch torchvision -q

# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

# Check versions
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("\n✅ Environment ready!")

---

## Section 1: Datasets & DataLoaders

### What is a Dataset?

`Dataset` is an abstract class representing a dataset. PyTorch provides built-in datasets like MNIST, CIFAR-10, and ImageNet.

**When to use it**: Loading standard datasets for computer vision tasks.

**Key methods**:
- `__len__()`: Returns the number of samples
- `__getitem__(idx)`: Returns a single sample (image, label)

In [ ]:
# Example: Loading CIFAR-10 dataset
# CIFAR-10: 60,000 32x32 color images in 10 classes

# Define a simple transform
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert PIL Image to tensor
])

# Load training dataset
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',           # Where to download data
    train=True,              # Training split
    download=True,           # Download if not present
    transform=transform      # Apply transformations
)

# Check dataset properties
print(f"Dataset size: {len(train_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")  # (C, H, W)
print(f"Label: {train_dataset[0][1]}")
print(f"Classes: {train_dataset.classes}")

### What is a DataLoader?

`DataLoader` wraps a `Dataset` to provide batching, shuffling, and parallel data loading.

**When to use it**: Training neural networks with mini-batch gradient descent.

**Key parameters**:
- `dataset`: The Dataset to load from
- `batch_size`: Number of samples per batch (default: 1)
- `shuffle`: Whether to shuffle data each epoch (default: False)
- `num_workers`: Number of subprocesses for data loading (default: 0)

In [ ]:
# Example: Creating a DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,      # 32 images per batch
    shuffle=True,       # Shuffle data each epoch
    num_workers=2       # 2 parallel workers (use 0 on Windows or for debugging)
)

# Iterate through batches
for images, labels in train_loader:
    print(f"Batch shape: {images.shape}")    # (batch_size, C, H, W)
    print(f"Labels shape: {labels.shape}")   # (batch_size,)
    print(f"Labels: {labels[:5]}")           # First 5 labels
    break  # Just show first batch

### Splitting Datasets: `random_split`

`random_split` splits a dataset into non-overlapping subsets (e.g., train/validation).

**When to use it**: Creating validation sets from training data.

**Returns**: A list of `Subset` objects that work with `DataLoader`.

In [ ]:
# Example: 80/20 train/validation split
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # For reproducibility
)

print(f"Training samples: {len(train_subset)}")
print(f"Validation samples: {len(val_subset)}")

# Create DataLoaders for each split
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

### Common Mistakes with Datasets & DataLoaders

❌ **Forgetting to shuffle training data**: Always use `shuffle=True` for training loaders

❌ **Shuffling validation data**: Validation loaders should have `shuffle=False` for reproducible metrics

❌ **Wrong num_workers on Windows**: Use `num_workers=0` on Windows to avoid multiprocessing issues

❌ **Forgetting to move data to device**: Remember `images = images.to(device)` in training loop

✅ **Typical pattern**:
```python
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
```

---

## Section 2: Image Transforms

### What are Transforms?

Transforms apply preprocessing and augmentation to images. PyTorch provides `torchvision.transforms` for common operations.

**When to use them**: Preparing images for neural networks (normalization, resizing, augmentation).

**Key concept**: Transforms are applied **before** feeding data to the model.

### `transforms.Compose()`: Chaining Transforms

`Compose` chains multiple transforms into a single pipeline.

**Parameters**: List of transforms to apply sequentially.

**Order matters**: Transforms are applied in the order specified.

In [ ]:
# Example: Basic transform pipeline
basic_transform = transforms.Compose([
    transforms.ToTensor(),                           # Convert PIL Image to tensor (0-1 range)
    transforms.Normalize(mean=[0.5, 0.5, 0.5],       # Normalize to (-1, 1)
                        std=[0.5, 0.5, 0.5])
])

# Load dataset with transforms
dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=basic_transform  # Applied to each image
)

# Check transformed image
img, label = dataset[0]
print(f"Transformed image shape: {img.shape}")
print(f"Pixel value range: [{img.min():.2f}, {img.max():.2f}]")

### Data Augmentation Transforms

Augmentation randomly modifies images to improve model generalization.

**Common augmentations**:
- `RandomHorizontalFlip(p=0.5)`: Randomly flip images horizontally
- `RandomCrop(size, padding)`: Randomly crop images
- `RandomRotation(degrees)`: Randomly rotate images
- `ColorJitter(brightness, contrast)`: Randomly change colors

**When to use them**: Training only (not validation/test).

In [ ]:
# Example: Training transforms with augmentation
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),          # 50% chance to flip
    transforms.RandomCrop(32, padding=4),            # Crop with padding
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # ImageNet statistics
                        std=[0.229, 0.224, 0.225])
])

# Validation transforms (no augmentation!)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Apply different transforms to train vs. validation
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=val_transform
)

print("✅ Augmentation applied to training data only")

### Common Transform Patterns

**Pattern 1: Basic preprocessing (no augmentation)**
```python
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[...], std=[...])
])
```

**Pattern 2: Training with augmentation**
```python
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(size, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[...], std=[...])
])
```

**Pattern 3: Validation (no augmentation, just preprocessing)**
```python
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[...], std=[...])
])
```

**Important**: `ToTensor()` must come before `Normalize()` because normalization operates on tensors, not PIL images.

---

## Section 3: CNN Layers

### `nn.Conv2d()`: 2D Convolution Layer

Applies 2D convolution over input images. Learns filters that detect features (edges, textures, patterns).

**Key parameters**:
- `in_channels`: Number of input channels (3 for RGB, 1 for grayscale)
- `out_channels`: Number of filters (output feature maps)
- `kernel_size`: Size of the convolutional kernel (e.g., 3 means 3×3)
- `stride`: Step size for sliding the kernel (default: 1)
- `padding`: Zero-padding added to input (default: 0)

**Output shape formula**:
```
H_out = floor((H_in + 2*padding - kernel_size) / stride) + 1
W_out = floor((W_in + 2*padding - kernel_size) / stride) + 1
```

In [ ]:
# Example: Conv2d layer
conv_layer = nn.Conv2d(
    in_channels=3,      # RGB input
    out_channels=16,    # 16 filters
    kernel_size=3,      # 3x3 kernel
    stride=1,
    padding=1           # Keep spatial dimensions (padding = (kernel_size - 1) / 2)
)

# Test with a batch of images (batch_size=4, channels=3, height=32, width=32)
sample_input = torch.randn(4, 3, 32, 32)
output = conv_layer(sample_input)

print(f"Input shape:  {sample_input.shape}")  # (4, 3, 32, 32)
print(f"Output shape: {output.shape}")         # (4, 16, 32, 32)
print(f"Number of parameters: {sum(p.numel() for p in conv_layer.parameters())}")
# Parameters = (kernel_size * kernel_size * in_channels * out_channels) + out_channels (bias)

### `nn.MaxPool2d()`: Max Pooling Layer

Downsamples feature maps by taking the maximum value in each pooling window. Reduces spatial dimensions and computational cost.

**Key parameters**:
- `kernel_size`: Size of the pooling window
- `stride`: Step size (default: same as kernel_size)
- `padding`: Zero-padding (default: 0)

**Common usage**: `MaxPool2d(2, 2)` reduces dimensions by half.

In [ ]:
# Example: MaxPool2d layer
pool_layer = nn.MaxPool2d(kernel_size=2, stride=2)

# Test with feature maps from conv layer
sample_input = torch.randn(4, 16, 32, 32)  # (batch, channels, height, width)
output = pool_layer(sample_input)

print(f"Input shape:  {sample_input.shape}")  # (4, 16, 32, 32)
print(f"Output shape: {output.shape}")         # (4, 16, 16, 16) - spatial dims halved

# MaxPool2d has no learnable parameters (it's just a max operation)
print(f"Number of parameters: {sum(p.numel() for p in pool_layer.parameters())}")  # 0

### Typical CNN Architecture Pattern

CNNs stack Conv → Activation → Pool blocks, followed by fully connected layers.

**Common pattern**:
```python
nn.Conv2d(...)      # Learn features
nn.ReLU()           # Non-linearity
nn.MaxPool2d(...)   # Downsample
# Repeat 2-5 times
nn.Flatten()        # Flatten for FC layers
nn.Linear(...)      # Classification
```

In [ ]:
# Example: Simple CNN architecture
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Block 1: 3 -> 32 channels, 32x32 -> 16x16
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Block 2: 32 -> 64 channels, 16x16 -> 8x8
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Fully connected: 64 * 8 * 8 = 4096 -> 10 classes
        self.fc = nn.Linear(64 * 8 * 8, 10)
    
    def forward(self, x):
        # Block 1
        x = self.pool1(F.relu(self.conv1(x)))  # (batch, 32, 16, 16)
        # Block 2
        x = self.pool2(F.relu(self.conv2(x)))  # (batch, 64, 8, 8)
        # Flatten and classify
        x = x.view(x.size(0), -1)              # (batch, 4096)
        x = self.fc(x)                          # (batch, 10)
        return x

# Test the model
model = SimpleCNN()
test_input = torch.randn(2, 3, 32, 32)  # Batch of 2 CIFAR-10 images
output = model(test_input)
print(f"Input:  {test_input.shape}")
print(f"Output: {output.shape}")  # (2, 10) - 10 class scores

### Quick Reference: Shape Calculations

| Operation | Input Shape | Output Shape | Formula |
|-----------|-------------|--------------|---------|
| `Conv2d(3, 64, 3, padding=1)` | (B, 3, H, W) | (B, 64, H, W) | Same size (padding=1) |
| `Conv2d(3, 64, 3, padding=0)` | (B, 3, H, W) | (B, 64, H-2, W-2) | Size decreases by 2 |
| `MaxPool2d(2, 2)` | (B, C, H, W) | (B, C, H/2, W/2) | Halves spatial dims |
| `Flatten()` or `view()` | (B, C, H, W) | (B, C*H*W) | Flattens feature maps |

**Key insight**: Convolutions preserve/reduce spatial size, pooling reduces size, channels are controlled by `out_channels`.

---

## Section 4: Pretrained Models & Transfer Learning

### Loading Pretrained Models from `torchvision.models`

PyTorch provides pretrained models trained on ImageNet (1.2M images, 1000 classes).

**When to use pretrained models**: When you have limited data or want faster training.

**Common architectures**:
- `resnet18`, `resnet50`: Residual networks (good balance of speed/accuracy)
- `vgg16`, `vgg19`: Very deep networks (slower but accurate)
- `mobilenet_v2`: Lightweight for mobile/edge devices

In [ ]:
# Example: Loading a pretrained ResNet18
from torchvision import models

# Load pretrained model
model = models.resnet18(pretrained=True)

# Check the model architecture
print(model)
print(f"\nOriginal final layer: {model.fc}")  # Linear(512, 1000) for ImageNet

### Modifying the Final Layer for Your Task

Pretrained models have 1000 output classes (ImageNet). For your task (e.g., CIFAR-10 with 10 classes), replace the final fully connected layer.

**Steps**:
1. Load pretrained model
2. Freeze earlier layers (optional, for feature extraction)
3. Replace final layer with new layer matching your number of classes
4. Train only the new layer (or fine-tune all layers)

In [ ]:
# Example: Adapt ResNet18 for CIFAR-10 (10 classes)

# Load pretrained model
model = models.resnet18(pretrained=True)

# Option 1: Freeze all layers except the final one (feature extraction)
for param in model.parameters():
    param.requires_grad = False  # Freeze all layers

# Replace final layer (only this layer will be trained)
num_features = model.fc.in_features  # Get input features to final layer
model.fc = nn.Linear(num_features, 10)  # 10 classes for CIFAR-10

print(f"Modified final layer: {model.fc}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# Option 2: Fine-tune all layers (don't freeze, train everything)
# Simply replace the final layer without freezing:
# model = models.resnet18(pretrained=True)
# model.fc = nn.Linear(model.fc.in_features, 10)
# All layers will be trainable

### `model.train()` vs `model.eval()`

**`model.train()`**: Sets the model to training mode
- Enables dropout (if present)
- Enables batch normalization updates
- Use during training

**`model.eval()`**: Sets the model to evaluation mode
- Disables dropout
- Uses running statistics for batch normalization
- Use during validation/testing

**Critical**: Always call these before training/evaluation loops!

In [ ]:
# Example: Using train() and eval() modes

model = SimpleCNN()

# During training
model.train()  # Enable dropout, batch norm updates
for images, labels in train_loader:
    # Forward pass, backward pass, optimization
    pass

# During validation/testing
model.eval()  # Disable dropout, use running stats
with torch.no_grad():  # Also disable gradient computation for speed
    for images, labels in val_loader:
        # Forward pass only
        pass

print("✅ Always use model.train() and model.eval() appropriately!")

### Typical Transfer Learning Workflow

**Pattern**: Load pretrained model → Modify final layer → Train/fine-tune

```python
# Step 1: Load pretrained model
model = models.resnet18(pretrained=True)

# Step 2 (Optional): Freeze earlier layers
for param in model.parameters():
    param.requires_grad = False

# Step 3: Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Step 4: Train only the new layer (or all layers if not frozen)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# ... training loop ...
```

**When to freeze layers**:
- **Freeze**: Small dataset, task similar to ImageNet
- **Fine-tune (don't freeze)**: Larger dataset, task different from ImageNet

---

## Additional Resources

### Official PyTorch Documentation
- [Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
- [Transforms](https://pytorch.org/vision/stable/transforms.html)
- [CNN Layers (nn.Conv2d, nn.MaxPool2d)](https://pytorch.org/docs/stable/nn.html#convolution-layers)
- [Pretrained Models](https://pytorch.org/vision/stable/models.html)

### Related Tutorials
- [Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [Data Augmentation](https://pytorch.org/vision/stable/auto_examples/plot_transforms.html)

### Back to Main Tutorial
Ready to practice? Go back to the [Week 2 CNN Tutorial](week_02_cnns_rnns.ipynb) and apply these concepts in hands-on labs!

---

**Questions?** Consult the official PyTorch documentation or ask your instructor.